In [142]:
!pip install pytorch-crf

In [143]:
import ast
import numpy as np
import pandas as pd
import tqdm as notebook_tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torchcrf import CRF
from sklearn.model_selection import train_test_split
from tqdm import tqdm

## Label maps

In [6]:
NER_TAGS = [
    'B-ACCOUNTNAME', 'B-ACCOUNTNUMBER', 'B-CREDITCARDNUMBER', 'B-EMAIL', 'B-IPV4', 'B-IPV6', 'B-MAC', 'B-PASSWORD', 'B-PHONE_NUMBER',
    'B-SSN', 'B-USERNAME',
    'I-ACCOUNTNAME', 'I-ACCOUNTNUMBER', 'I-CREDITCARDNUMBER', 'I-EMAIL', 'I-IPV4', 'I-IPV6', 'I-MAC', 'I-PASSWORD', 'I-PHONE_NUMBER',
    'I-SSN', 'I-USERNAME', 'O'
]
tag2idx = {tag: i for i, tag in enumerate(NER_TAGS)}
idx2tag = {i: tag for tag, i in tag2idx.items()}
NUM_TAGS = len(NER_TAGS)

## Tokenizer

The dataset was pre-tokenized with `distilbert-base-uncased`, so we load
the same tokenizer here — only to call `convert_tokens_to_ids` and to
obtain the correct `pad_token_id` (0).  We do **not** re-tokenize.

In [145]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

## Load data

In [146]:
df = pd.read_parquet('/kaggle/input/datasets/karimmahmoud09/pii-safety-dataset/transformed_train.parquet')
df = pd.DataFrame(df)
df.head()

,masked_text,unmasked_text,token_entity_labels,tokenised_unmasked_text
0,[PREFIX_1] [FIRSTNAME_1] [MIDDLENAME_1] [LASTN...,"Mr. Adolphus Reagan Ziemann, as a Central Prin...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[mr, ., adolph, ##us, reagan, z, ##ie, ##mann,..."
1,"Hello [FIRSTNAME_1], would you please investig...","Hello Hannah, would you please investigate the...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[hello, hannah, ,, would, you, please, investi..."
2,We also request a review of our policies with ...,We also request a review of our policies with ...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[we, also, request, a, review, of, our, polici..."
3,"Dear [FIRSTNAME_1], a company-wide presentatio...","Dear Devan, a company-wide presentation is req...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[dear, dev, ##an, ,, a, company, -, wide, pres..."
4,Can we also have a session on how to manage st...,Can we also have a session on how to manage st...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[can, we, also, have, a, session, on, how, to,..."


In [147]:
# Drop rows where token list and label list have different lengths
before = len(df)
df = df[
    df.apply(
        lambda r: len(r["tokenised_unmasked_text"]) == len(r["token_entity_labels"]),
        axis=1
    )
].reset_index(drop=True)
after = len(df)
print(f"Removed {before - after} rows")

Removed 190 rows


In [148]:
df = df[
    df["tokenised_unmasked_text"].apply(len) > 0
].reset_index(drop=True)

In [149]:
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=42)
print(len(train_df), len(val_df), len(test_df))

17330 1926 2140


In [150]:
tokens = train_df.iloc[0]["tokenised_unmasked_text"]
ids    = tokenizer.convert_tokens_to_ids(tokens)
print(tokens[:10])
print(ids[:10])

['dear' 'hr' ',' 'please' 'initiate' 'the' 'process' 'to' 'update' 'the']
[6203, 17850, 1010, 3531, 17820, 1996, 2832, 2000, 10651, 1996]


In [151]:
all_tags = set()
for labels in df["token_entity_labels"]:
    all_tags.update(labels)

for tag in NER_TAGS:
    if tag not in all_tags:
        print(f"Tag {tag} not found in dataset!")

In [152]:
from collections import Counter

counter = Counter()
for labels in train_df["token_entity_labels"]:
    counter.update(labels)
print(counter.most_common())

[('O', 1044469), ('I-IPV6', 23476), ('I-EMAIL', 9802), ('I-MAC', 9426), ('I-PHONE_NUMBER', 8686), ('I-CREDITCARDNUMBER', 6698), ('I-PASSWORD', 6438), ('I-IPV4', 5646), ('I-USERNAME', 3946), ('I-SSN', 3551), ('I-ACCOUNTNUMBER', 3317), ('I-ACCOUNTNAME', 1413), ('B-EMAIL', 1167), ('B-USERNAME', 1055), ('B-IPV4', 941), ('B-ACCOUNTNUMBER', 904), ('B-PHONE_NUMBER', 870), ('B-PASSWORD', 861), ('B-ACCOUNTNAME', 852), ('B-IPV6', 800), ('B-CREDITCARDNUMBER', 782), ('B-MAC', 746), ('B-SSN', 621)]


## Model


In [153]:


class DistilBERTBiLSTM(nn.Module):

    def __init__(self, num_tags, ignore_index=-100):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(512, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        self.ignore_index = ignore_index

    def forward(self, input_ids, attention_mask, labels=None):

        bert_out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask.float()
        )

        lstm_out, _ = self.lstm(bert_out.last_hidden_state)

        emissions = self.classifier(lstm_out)   # (B, T, num_tags)

        mask = attention_mask.bool()            # (B, T)

        if labels is not None:
            # FIX: CRF indexes emissions with label values directly.
            # Pad positions have label = -100 which causes an index error
            # even when masked. Clamp to 0 — the mask will ignore them anyway.
            safe_labels = labels.clone()
            safe_labels[labels == self.ignore_index] = 0

            loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            return loss, emissions

      
        predictions = self.crf.decode(emissions, mask=mask)
        return predictions

## Dataset and DataLoader


In [154]:
from torch.utils.data import Dataset

class PIIBertDataset(Dataset):

    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        tokens    = row["tokenised_unmasked_text"]
        labels    = row["token_entity_labels"]
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        label_ids = [tag2idx[t] for t in labels]

        return input_ids, label_ids

In [155]:
import torch
from torch.nn.utils.rnn import pad_sequence

IGNORE_INDEX = -100

def collate_fn(batch):
    token_ids = [torch.tensor(x[0], dtype=torch.long) for x in batch]
    label_ids = [torch.tensor(x[1], dtype=torch.long) for x in batch]

    token_ids = pad_sequence(token_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    label_ids = pad_sequence(label_ids, batch_first=True, padding_value=IGNORE_INDEX)

    mask = (token_ids != tokenizer.pad_token_id)

    return token_ids, label_ids, mask

In [156]:
from torch.utils.data import DataLoader

train_dataset = PIIBertDataset(train_df)
val_dataset   = PIIBertDataset(val_df)
test_dataset  = PIIBertDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, collate_fn=collate_fn)

In [157]:
# Shape sanity-check
tokens, labels, mask = next(iter(train_loader))
print("tokens:", tokens.shape)
print("labels:", labels.shape)
print("mask  :", mask.shape)

tokens: torch.Size([32, 137])
labels: torch.Size([32, 137])
mask  : torch.Size([32, 137])


## Training utilities

In [158]:
def overfit_one_batch(model, dataset, batch_size=16, epochs=200, lr=1e-4):
    loader     = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    x, y, mask = next(iter(loader))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    x, y, mask = x.to(device), y.to(device), mask.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        loss, emissions = model(input_ids=x, attention_mask=mask, labels=y)  # FIX: was `logits`
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            # FIX: crf.decode returns List[List[int]], iterate per sentence
            preds_list = model.crf.decode(emissions, mask=mask.bool())
            correct = 0
            total   = 0
            for i, pred_seq in enumerate(preds_list):
                seq_len = len(pred_seq)
                pred_t  = torch.tensor(pred_seq, device=device)
                true_t  = y[i, :seq_len]
                v       = (true_t != IGNORE_INDEX)
                correct += (pred_t[v] == true_t[v]).sum().item()
                total   += v.sum().item()
            acc = correct / total if total > 0 else 0.0
            print(f"Epoch {epoch:3d} | Loss {loss.item():.4f} | Acc {acc:.4f}")

In [159]:
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    correct    = 0
    total      = 0

    with torch.no_grad():
        for x, y, mask in tqdm(loader, desc="Evaluating"):
            x, y, mask = x.to(device), y.to(device), mask.to(device)

            loss, emissions = model(input_ids=x, attention_mask=mask, labels=y)
            total_loss += loss.item()

            # decode returns List[List[int]], pad to tensor for comparison
            preds_list = model.crf.decode(emissions, mask=mask.bool())
            valid = mask & (y != -100)

            for i, pred_seq in enumerate(preds_list):
                seq_len = len(pred_seq)
                pred_t  = torch.tensor(pred_seq, device=device)
                true_t  = y[i, :seq_len]
                v       = valid[i, :seq_len]
                correct += (pred_t[v] == true_t[v]).sum().item()
                total   += v.sum().item()

    return total_loss / len(loader), correct / total if total > 0 else 0.0

In [160]:
def train(
    model,
    train_dataset,
    val_dataset,
    batch_size=16,
    epochs=5,
    lr=2e-5,
    patience=2
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    # FIX: added crf param group — CRF transition matrix must be trained too
    optimizer = torch.optim.AdamW([
        {"params": model.bert.parameters(),       "lr": lr},
        {"params": model.lstm.parameters(),       "lr": lr * 10},
        {"params": model.classifier.parameters(), "lr": lr * 10},
        {"params": model.crf.parameters(),        "lr": lr * 10},
    ])

    best_loss  = float("inf")
    no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for x, y, mask in tqdm(train_loader, desc=f"Epoch {epoch}"):
            x, y, mask = x.to(device), y.to(device), mask.to(device)
            optimizer.zero_grad()
            loss, _ = model(input_ids=x, attention_mask=mask, labels=y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        train_loss       = total_loss / len(train_loader)
        val_loss, val_acc = evaluate(model, val_loader, device)

        print(
            f"Epoch {epoch:2d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_loss < best_loss:
            best_loss  = val_loss
            no_improve = 0
            torch.save(model.state_dict(), "best.pt")
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping")
                break

    model.load_state_dict(torch.load("best.pt"))
    return model




## Instantiate model

In [161]:
model = DistilBERTBiLSTM(num_tags=NUM_TAGS)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [162]:
# FIX: move model to device BEFORE running the forward pass
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

with torch.no_grad():
    x, y, mask = next(iter(val_loader))
    x    = x.to(device)
    mask = mask.to(device)

    preds_list = model(input_ids=x, attention_mask=mask)


print("Unique predicted tag indices:", torch.unique(preds))

Unique predicted tag indices: tensor([ 0,  1,  2,  4,  5,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20,
        21, 22, 23, 24], device='cuda:0')


## Optional: overfit sanity-check

Uncomment to verify the model can memorise one batch before full training.

In [163]:
# dataset = PIIBertDataset(train_df)
# overfit_one_batch(model, dataset, batch_size=16, epochs=300, lr=1e-4)

## Train

`lr=2e-5` is the standard fine-tuning rate for DistilBERT.  The BiLSTM
and classifier heads receive `lr * 10 = 2e-4` via the param groups above.

In [164]:
trained_model = train(
    model,
    train_dataset,
    val_dataset,
    batch_size=32,
    epochs=20,
    lr=2e-5,       
    patience=3
)

Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.44it/s]


Epoch  1 | Train Loss: 8.7729 | Val Loss: 2.3515 | Val Acc: 0.9804


Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.45it/s]


Epoch  2 | Train Loss: 1.9642 | Val Loss: 1.5469 | Val Acc: 0.9859


Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.43it/s]


...

Epoch  4 | Train Loss: 1.2528 | Val Loss: 1.5919 | Val Acc: 0.9859


Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.43it/s]


Epoch  5 | Train Loss: 1.0806 | Val Loss: 1.8490 | Val Acc: 0.9883


Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.44it/s]


Epoch  6 | Train Loss: 0.8538 | Val Loss: 2.4992 | Val Acc: 0.9886
Early stopping


## Evaluation

In [165]:
from tqdm import tqdm
from sklearn.metrics import classification_report
def test_evaluate(model, test_loader, device):
    model.eval()
    sent_preds  = []
    sent_labels = []

    with torch.no_grad():
        for x, y, mask in tqdm(test_loader, desc="Testing"):
            x, y, mask = x.to(device), y.to(device), mask.to(device)

            preds_list = model(input_ids=x, attention_mask=mask)

            for i, pred_seq in enumerate(preds_list):
                seq_len   = len(pred_seq)
                true_list = y[i, :seq_len].cpu().tolist()
                sent_preds.append(pred_seq)
                sent_labels.append(true_list)

    return sent_labels, sent_preds

In [166]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sent_labels, sent_preds = test_evaluate(trained_model, test_loader, device)

# Flatten for sklearn token-level report
flat_labels = [t for seq in sent_labels for t in seq]
flat_preds  = [t for seq in sent_preds  for t in seq]

print(classification_report(
    flat_labels,
    flat_preds,
    labels=list(range(NUM_TAGS)),
    target_names=[idx2tag[i] for i in range(NUM_TAGS)],
    zero_division=0
))

Testing: 100%|██████████| 67/67 [00:11<00:00,  5.77it/s]


                    precision    recall  f1-score   support

     B-ACCOUNTNAME       0.95      0.99      0.97       105
   B-ACCOUNTNUMBER       0.99      0.99      0.99       111
B-CREDITCARDNUMBER       0.74      0.94      0.83        89
           B-EMAIL       0.99      0.99      0.99       156
            B-IPV4       0.73      1.00      0.85       126
            B-IPV6       0.72      0.97      0.83       107
             B-MAC       1.00      1.00      1.00        84
        B-PASSWORD       1.00      0.89      0.94       102
    B-PHONE_NUMBER       1.00      1.00      1.00       107
             B-SSN       1.00      0.99      0.99        93
        B-USERNAME       0.95      0.95      0.95       147
     I-ACCOUNTNAME       0.95      0.99      0.97       179
   I-ACCOUNTNUMBER       1.00      0.99      1.00       388
I-CREDITCARDNUMBER       0.74      0.95      0.83       758
           I-EMAIL       1.00      1.00      1.00      1258
            I-IPV4       0.73      1.00

In [167]:
!pip install seqeval -q

In [168]:
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score


true_tags = [[idx2tag[t] for t in seq] for seq in sent_labels]
pred_tags = [[idx2tag[t] for t in seq] for seq in sent_preds]

print(seqeval_report(true_tags, pred_tags))
print("Span-level F1:", f1_score(true_tags, pred_tags))

                  precision    recall  f1-score   support

     ACCOUNTNAME       0.95      0.99      0.97       105
   ACCOUNTNUMBER       0.97      0.98      0.98       111
CREDITCARDNUMBER       0.73      0.94      0.82        89
           EMAIL       0.99      0.99      0.99       156
            IPV4       0.72      0.99      0.84       126
            IPV6       0.69      0.97      0.81       107
             MAC       0.99      0.99      0.99        84
        PASSWORD       1.00      0.89      0.94       102
    PHONE_NUMBER       1.00      1.00      1.00       107
             SSN       0.99      1.00      0.99        93
        USERNAME       0.95      0.95      0.95       147

       micro avg       0.89      0.97      0.93      1227
       macro avg       0.91      0.97      0.94      1227
    weighted avg       0.91      0.97      0.94      1227

Span-level F1: 0.9314107560405299


In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('test.parquet')
df = pd.DataFrame(df)
df.head()

,masked_text,unmasked_text,token_entity_labels,tokenised_unmasked_text
0,Hello [PREFIX_1] [FIRSTNAME_1]. Could you plea...,Hello Ms. Juliet. Could you please send me the...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[hello, ms, ., juliet, ., could, you, please, ..."
1,"To make sure we follow every protocol, we advi...","To make sure we follow every protocol, we advi...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[to, make, sure, we, follow, every, protocol, ..."
2,"Hello, our team is conducting a study involvin...","Hello, our team is conducting a study involvin...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[hello, ,, our, team, is, conducting, a, study..."
3,J'ai un client dont les biens personnels ont é...,J'ai un client dont les biens personnels ont é...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[j, ', ai, un, client, don, ##t, les, bien, ##..."
4,Does the policy on [CREDITCARDCVV_1] storage c...,Does the policy on 469 storage comply with cur...,"[O, O, O, O, O, O, O, O, O, O, O, O, O]","[does, the, policy, on, 46, ##9, storage, comp..."


In [ ]:
from pii_inference import PIIDetector , mask_pii


checkpoint = "models/best_bert_bilstm_crf.pt"
detector   = PIIDetector(checkpoint)

d:\Apps\anaconda\envs\langgraph-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 248.36it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from sklearn.metrics import classification_report
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score

true_labels = []   
pred_labels = []  

true_seqs   = []   
pred_seqs   = []

skipped = 0

for text, true_tags in zip(df["unmasked_text"], df["token_entity_labels"]):
    words, subword_ids, alignment = detector.tokenise_with_alignment(text)

    if len(subword_ids) != len(true_tags):
        skipped += 1
        continue

    tagged = detector.predict(text)   

    sent_true = []
    sent_pred = []

    for word_idx, subword_idx in enumerate(alignment):
        true_tag = true_tags[subword_idx]          
        pred_tag = tagged[word_idx][1]             

        true_labels.append(tag2idx[true_tag])
        pred_labels.append(tag2idx[pred_tag])
        sent_true.append(true_tag)
        sent_pred.append(pred_tag)

    true_seqs.append(sent_true)
    pred_seqs.append(sent_pred)

print(f"Skipped {skipped} misaligned rows")
print(f"Evaluated {len(true_seqs)} sentences, {len(true_labels)} words\n")

accuracy = sum(t == p for t, p in zip(true_labels, pred_labels)) / len(true_labels)
print(f"Word-level accuracy: {accuracy:.4f}\n")

print("── Token / Word-Level Report (sklearn) ──\n")
print(classification_report(         
    true_labels,
    pred_labels,
    labels=list(range(NUM_TAGS)),
    target_names=[idx2tag[i] for i in range(NUM_TAGS)],
    zero_division=0
))

print("── Span-Level Report (seqeval) ──\n")
print(seqeval_report(true_seqs, pred_seqs))
print("Span-Level F1:", f1_score(true_seqs, pred_seqs))